In [1]:
with open('data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
print(len(text))

1115394


In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [4]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [5]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
data = torch.tensor(encode(text), dtype=torch.long, device=device)
print(data.shape, data.dtype)
print(data[:100])

cuda
torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59], device='cuda:0')


In [6]:
n = int(len(data)*0.9)
train_data = data[:n]
val_data = data[n:]

In [7]:
context_len = 8
train_data[:context_len+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58], device='cuda:0')

In [8]:
x = train_data[:context_len]
y = train_data[1:context_len+1]
for t in range(context_len):
    context = x[:t+1]
    target = y[t]
    print(f"input: {context} - target: {target}")

input: tensor([18], device='cuda:0') - target: 47
input: tensor([18, 47], device='cuda:0') - target: 56
input: tensor([18, 47, 56], device='cuda:0') - target: 57
input: tensor([18, 47, 56, 57], device='cuda:0') - target: 58
input: tensor([18, 47, 56, 57, 58], device='cuda:0') - target: 1
input: tensor([18, 47, 56, 57, 58,  1], device='cuda:0') - target: 15
input: tensor([18, 47, 56, 57, 58,  1, 15], device='cuda:0') - target: 47
input: tensor([18, 47, 56, 57, 58,  1, 15, 47], device='cuda:0') - target: 58


In [9]:
torch.manual_seed(1337)
batch_size = 4
context_len = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - context_len, (batch_size,))
    x = torch.stack([data[i:i+context_len] for i in ix])
    y = torch.stack([data[i+1:i+context_len+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)
print('-------------')

for b in range(batch_size):
    for t in range(context_len):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"input: {context} - target: {target}")


inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]], device='cuda:0')
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]], device='cuda:0')
-------------
input: tensor([24], device='cuda:0') - target: 43
input: tensor([24, 43], device='cuda:0') - target: 58
input: tensor([24, 43, 58], device='cuda:0') - target: 5
input: tensor([24, 43, 58,  5], device='cuda:0') - target: 57
input: tensor([24, 43, 58,  5, 57], device='cuda:0') - target: 1
input: tensor([24, 43, 58,  5, 57,  1], device='cuda:0') - target: 46
input: tensor([24, 43, 58,  5, 57,  1, 46], device='cuda:0') - target: 43
input: tensor([24, 43, 58,  5, 57,  1, 46, 43], device='cuda:0') - target: 39
input: tensor([44], device='cuda:0') - target: 53

In [10]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # (B, T, C)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:,-1,:] # (B, C)
            probs = F.softmax(logits, dim=-1) # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat((idx,idx_next), dim=1)  # (B, T+1)
        return idx
    
m = BigramLanguageModel(vocab_size)
m.to(device=device)
out, loss = m(xb, yb)
print(out.shape, loss)

torch.Size([32, 65]) tensor(4.8786, device='cuda:0', grad_fn=<NllLossBackward0>)


In [11]:
print(out[0])

tensor([-1.5101, -0.0948,  1.0927,  0.1505,  1.6347, -0.0518,  0.4996,  0.7216,
        -0.8968, -0.4122,  1.0030,  0.8508,  0.2178,  0.0328, -0.1699,  1.0659,
        -0.6177,  1.1824,  0.0214, -0.2154, -1.4623,  2.1707,  0.1624,  1.0296,
         0.4154,  0.6207,  0.2341, -0.0326,  1.0124,  1.5122, -0.3359,  0.2456,
         1.8682,  0.7536, -0.1177, -0.1967, -0.9552, -0.8995, -0.9583, -0.5945,
         0.1321, -0.5406,  0.1405, -0.7321,  1.1796,  1.3316, -0.2094,  0.0960,
         0.9040, -0.4032,  0.3027, -0.8034, -1.2537, -1.5195,  0.7446,  1.1914,
        -0.8061, -0.6290,  1.2447, -2.4400,  0.8408, -0.3993, -0.6126, -0.6597,
         0.7624], device='cuda:0', grad_fn=<SelectBackward0>)


In [12]:
print(decode(m.generate(torch.zeros((1, 1), dtype=torch.long, device=device), max_new_tokens=100)[0].tolist()))


pYCXxfRkRZd
wc'wfNfT;OLlTEeC K
jxqPToTb?bXAUG:C-SGJO-33SM:C?YI3a
hs:LVXJFhXeNuwqhObxZ.tSVrddXlaSZaNe


In [13]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)


In [21]:
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
print(loss.item())


2.6078572273254395


In [22]:
print(decode(m.generate(torch.zeros((1, 1), dtype=torch.long, device=device), max_new_tokens=100)[0].tolist()))


MPUESTh,
Mad
Whed my o myr f-NLIERor,
SS&y, wardsal thes ghesthidin cour ay aney Iry ts I fr y ce.
J


Mathematical trick

In [52]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2 # batch, time, chanels
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [53]:
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]
        xbow[b,t] = torch.mean(xprev,0)

In [56]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x
torch.allclose(xbow, xbow2, atol=1e-6)

True

In [57]:
trill = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(trill==0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3, atol=1e-6)

True

In [50]:
xbow2.shape

torch.Size([4, 8, 2])

In [31]:
torch.tril(torch.ones(3,3))

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

In [39]:
torch.manual_seed(42)

a = torch.tril(torch.ones(3,3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0, 10, (3,2)).float()
c = a @ b
print(a)
print(b)
print(c)

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])
